# Lab 2 — Finding yourself with a magnetometer

You walked a 24-metre path: 16 m of corridor, a 90° turn, 8 m of atrium, with a marker every 2 m.

In this notebook you will turn your **mapping walk** into a magnetic map of that path, and then ask a
simple question about your **test walks**: *given a few seconds of magnetometer, where was I?*

You do not have to write any code. Every section has **one number to change**. Change it, run the cell,
look at what happens.


## Setup

Run the cell below once. It loads the pilot recordings so that everything works immediately.

If you want to use **your own** recordings instead, run the second cell and upload your Sensor Logger
`.zip` files.

In [ ]:
#@title Functions — you do not need to open this cell { display-mode: "form" }
# Everything below is plumbing: reading the CSV files, finding the pauses, building the map,
# sliding the window, drawing. The lesson is in the cells further down.

import base64, io, os, zipfile
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view

_PILOT = "..."
if not os.path.isdir("pilot_data"):
    zipfile.ZipFile(io.BytesIO(base64.b64decode(_PILOT))).extractall("pilot_data")


DATA = "pilot_data"
MARKERS = np.arange(0.0, 25.0, 2.0)
PATH_LENGTH = 24.0
LANDMARKS = {14: "fire doors", 16: "90 corner", 18: "lift", 20: "goods lift", 24: "entrance"}
STEP = 0.05
GRID = np.arange(0.0, PATH_LENGTH + 1e-9, STEP)
plt.rcParams.update({"figure.figsize": (13, 4.5), "axes.grid": True, "grid.alpha": .25,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})

# ---------- loading ----------
def _read(rec, sensor):
    d = pd.read_csv(f"{DATA}/{rec}/{sensor}.csv")
    return d.seconds_elapsed.values, np.sqrt(d.x.values**2 + d.y.values**2 + d.z.values**2)

def _magnetometer(rec, which="calibrated"):
    return _read(rec, "Magnetometer" if which == "calibrated" else "MagnetometerUncalibrated")

def _movement(rec, window_s=1.0):
    """How much the phone is moving: rolling std of |acceleration|."""
    t, a = _read(rec, "Accelerometer")
    n = max(3, int(round(window_s / np.median(np.diff(t)))))
    return t, pd.Series(a).rolling(n, center=True).std().bfill().ffill().values

def _runs(t, mask, min_len):
    out, i = [], 0
    while i < len(mask):
        if mask[i]:
            j = i
            while j < len(mask) and mask[j]:
                j += 1
            if t[j - 1] - t[i] >= min_len:
                out.append((t[i], t[j - 1]))
            i = j
        else:
            i += 1
    return out

def find_pauses(rec, threshold=0.5):
    t, mv = _movement(rec)
    return _runs(t, mv < threshold * np.median(mv), 0.4)

def find_walking(rec, threshold=0.5):
    t, mv = _movement(rec)
    runs = _runs(t, mv > threshold * np.median(mv), 3.0)
    return runs[0][0], runs[-1][1]

# ---------- the map ----------
def build_map(rec="01_mapping", threshold=0.5, magnetometer="calibrated"):
    """Mapping walk -> |B| as a function of distance along the path."""
    pauses = find_pauses(rec, threshold)
    if len(pauses) != len(MARKERS):
        raise ValueError(f"Found {len(pauses)} pauses but the path has {len(MARKERS)} markers. "
                         f"Adjust SENSITIVITY and look at the plot above.")
    t, B = _magnetometer(rec, magnetometer)
    ts, ds = [], []
    for (t0, t1), d in zip(pauses, MARKERS):
        ts += [t0, t1]; ds += [d, d]          # standing still on a marker = distance does not change
    distance = np.interp(t, ts, ds)
    keep = (t >= pauses[0][0]) & (t <= pauses[-1][1])
    d, b = distance[keep], B[keep]
    o = np.argsort(d)
    return np.interp(GRID, d[o], b[o])

def load_walk(rec, magnetometer="calibrated"):
    """Test walk -> |B| as a function of distance, assuming a constant pace between the two ends."""
    t0, t1 = find_walking(rec)
    t, B = _magnetometer(rec, magnetometer)
    keep = (t >= t0) & (t <= t1)
    d, b = (t[keep] - t0) / (t1 - t0) * PATH_LENGTH, B[keep]
    if "reverse" in rec:
        d = PATH_LENGTH - d
    o = np.argsort(d)
    return np.interp(GRID, d[o], b[o])

# ---------- matching ----------
def _windows(profile, n):
    return sliding_window_view(profile, n)

def _match_one(map_profile, window, compare):
    segments = _windows(map_profile, len(window))
    if compare == "shape":
        segments = segments - segments.mean(axis=1, keepdims=True)
        window = window - window.mean()
    scores = ((segments - window) ** 2).sum(axis=1)
    return scores                                  # one number per candidate position

def _candidate_positions(n_win):
    return GRID[n_win // 2: n_win // 2 + len(_windows(GRID, n_win))]

def locate(map_profile, walk_profile, W=3.0, compare="values", every=0.25):
    """Slide a W-metre window of the walk along the map. Returns (true, estimated)."""
    n = max(3, int(round(W / STEP)) | 1)
    half = n // 2
    true, est = [], []
    cand = _candidate_positions(n)
    for i in range(half, len(GRID) - half, max(1, int(round(every / STEP)))):
        scores = _match_one(map_profile, walk_profile[i - half:i + half + 1], compare)
        true.append(GRID[i]); est.append(cand[np.argmin(scores)])
    return np.array(true), np.array(est)

def error(map_profile, walk_profile, W=3.0, compare="values"):
    t, e = locate(map_profile, walk_profile, W, compare)
    return np.median(np.abs(t - e))

# ---------- plots ----------
def _mark_landmarks(ax):
    lo, hi = ax.get_ylim()
    ax.set_ylim(lo, hi + 0.16 * (hi - lo))          # headroom so the labels clear the title
    for d, name in LANDMARKS.items():
        ax.axvline(d, color="grey", ls="--", lw=.8, zorder=0)
        ax.annotate(name, (d, hi + 0.02 * (hi - lo)),
                    ha="center", va="bottom", fontsize=8, color="grey")

def plot_pauses(rec="01_mapping", threshold=0.5):
    t, mv = _movement(rec)
    pauses = find_pauses(rec, threshold)
    fig, ax = plt.subplots()
    ax.plot(t, mv, lw=.8, color="#333")
    ax.axhline(threshold * np.median(mv), color="crimson", lw=1, label="threshold")
    for t0, t1 in pauses:
        ax.axvspan(t0, t1, color="crimson", alpha=.25)
    ax.set_yscale("log"); ax.set_xlabel("time (s)"); ax.set_ylabel("how much the phone moves")
    ax.set_title(f"{len(pauses)} pauses found — the path has {len(MARKERS)} markers")
    ax.legend(); plt.show()
    print(f"Found {len(pauses)} pauses. You need exactly {len(MARKERS)}.")

def plot_map(map_profile):
    fig, ax = plt.subplots()
    ax.plot(GRID, map_profile, lw=1.5, color="#1f4e79")
    ax.plot(MARKERS, np.interp(MARKERS, GRID, map_profile), "ko", ms=4)
    ax.set_xlabel("distance along the path (m)"); ax.set_ylabel("|B|  (µT)")
    ax.set_title("Your magnetic map")
    _mark_landmarks(ax); plt.show()

def how_many_positions(map_profile, reading):
    # smooth a little first, otherwise sensor noise wiggling across the line counts as many crossings
    smooth = pd.Series(map_profile).rolling(int(0.5 / STEP), center=True).mean().bfill().ffill().values
    found = np.where(np.diff(np.sign(smooth - reading)) != 0)[0]
    crossings = [i for k, i in enumerate(found) if k == 0 or (i - found[k - 1]) * STEP > 0.5]
    fig, ax = plt.subplots()
    ax.plot(GRID, map_profile, lw=1.5, color="#1f4e79")
    ax.axhline(reading, color="crimson", lw=1.5)
    ax.plot(GRID[crossings], np.full(len(crossings), reading), "o", color="crimson", ms=8)
    ax.set_xlabel("distance (m)"); ax.set_ylabel("|B|  (µT)")
    ax.set_title(f"A single reading of {reading} µT matches {len(crossings)} different positions")
    plt.show()
    print(f"{reading} µT -> {len(crossings)} possible positions.")

def plot_location(map_profile, walk_profile, W=3.0, compare="values"):
    t, e = locate(map_profile, walk_profile, W, compare)
    err = np.abs(t - e)
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    ax[0].plot([0, PATH_LENGTH], [0, PATH_LENGTH], "k--", lw=.8)
    ax[0].plot(t, e, ".", ms=6, color="#1f4e79")
    ax[0].set_xlabel("true position (m)"); ax[0].set_ylabel("estimated position (m)")
    ax[0].set_aspect("equal"); ax[0].set_xlim(0, PATH_LENGTH); ax[0].set_ylim(0, PATH_LENGTH)
    ax[1].plot(t, err, lw=1.2, color="crimson"); ax[1].axhline(1, color="green", ls=":", lw=1)
    ax[1].set_yscale("symlog", linthresh=1)
    ax[1].set_xlabel("true position (m)"); ax[1].set_ylabel("error (m)")
    fig.suptitle(f"window = {W} m,  comparing {compare}   ->   median error {np.median(err):.2f} m")
    plt.tight_layout(); plt.show()
    print(f"Median error: {np.median(err):.2f} m     within 1 m: {100*(err<1).mean():.0f}% of the path")

def plot_error_vs_window(map_profile, walk_profile, windows=(0.1, 0.5, 1, 1.5, 2, 3, 4, 5, 6, 8, 10),
                         compare="values"):
    errs = [error(map_profile, walk_profile, w, compare) for w in windows]
    fig, ax = plt.subplots()
    ax.plot(windows, errs, "o-", color="#1f4e79")
    ax.set_xlabel("window length W (m)"); ax.set_ylabel("median error (m)")
    ax.set_title("How much does the window length matter?")
    plt.show()
    for w, e in zip(windows, errs):
        print(f"  W = {w:4} m  ->  {e:5.2f} m")

def plot_ambiguity(map_profile, walk_profile, W=3.0, min_separation=3.0):
    """For each place on the map: how similar is its best look-alike somewhere far away?"""
    n = max(3, int(round(W / STEP)) | 1); half = n // 2
    cand = _candidate_positions(n)
    margin = []
    for i in range(half, len(GRID) - half):
        scores = _match_one(map_profile, map_profile[i - half:i + half + 1], "values")
        margin.append(scores[np.abs(cand - GRID[i]) > min_separation].min())
    pos, margin = GRID[half:len(GRID) - half], np.array(margin)
    t, e = locate(map_profile, walk_profile, W)
    fig, ax = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
    ax[0].plot(pos, margin, color="purple", lw=1.3); ax[0].set_yscale("log")
    ax[0].set_ylabel("how unique\nthis place is")
    ax[0].set_title("Where the map has look-alikes (top) and where we actually get it wrong (bottom)")
    ax[1].plot(t, np.abs(t - e), color="crimson", lw=1.2); ax[1].axhline(1, color="green", ls=":", lw=1)
    ax[1].set_yscale("symlog", linthresh=1)
    ax[1].set_ylabel("error (m)"); ax[1].set_xlabel("distance (m)")
    plt.tight_layout(); plt.show()

def where_am_i(map_profile, rec, at_second, assumed_speed=0.7, W=3.0):
    """Locate ONE fragment of a recording, without knowing where the walk started or ended."""
    t, B = _magnetometer(rec)
    span = (W / 2) / assumed_speed
    sel = (t >= at_second - span) & (t <= at_second + span)
    if sel.sum() < 10:
        print("That instant is outside the recording."); return None
    d = (t[sel] - t[sel][0]) * assumed_speed
    xs = np.arange(0, W + 1e-9, STEP)
    window = np.interp(xs, d, B[sel])
    scores = _match_one(map_profile, window, "values")
    cand = _candidate_positions(len(window))
    answer = cand[np.argmin(scores)]
    fig, ax = plt.subplots(2, 1, figsize=(13, 6))
    ax[0].plot(GRID, map_profile, lw=1.3, color="#1f4e79", label="the map")
    ax[0].plot(answer + xs - W / 2, window, lw=2.5, color="crimson", label="your fragment, placed here")
    ax[0].legend(fontsize=9); ax[0].set_ylabel("|B| (µT)")
    ax[1].plot(cand, scores, color="purple", lw=1.3); ax[1].set_yscale("log")
    ax[1].axvline(answer, color="crimson", lw=1.4)
    ax[1].set_xlabel("candidate position on the map (m)"); ax[1].set_ylabel("disagreement")
    fig.suptitle(f"{W} m of recording around second {at_second}, assuming {assumed_speed} m/s  ->  you were at {answer:.1f} m")
    plt.tight_layout(); plt.show()
    print(f"That fragment was recorded at {answer:.1f} m along the path.")

def use_my_recordings():
    """Upload your own Sensor Logger .zip files (you can pick several at once)."""
    global DATA
    from google.colab import files
    os.makedirs("my_data", exist_ok=True)
    for name, blob in files.upload().items():
        folder = os.path.join("my_data", os.path.splitext(name)[0].split("-")[0])
        zipfile.ZipFile(io.BytesIO(blob)).extractall(folder)
    DATA = "my_data"
    print("\nRecordings now available:")
    for f in sorted(os.listdir(DATA)):
        print("   ", f)
    print("\nCopy the two names you need into MAPPING_WALK and TEST_WALK below.")

print("Ready. Using the pilot recordings from the corridor outside room E2.")


In [ ]:
# Only if you want to use your own recordings:
# use_my_recordings()

MAPPING_WALK = "01_mapping"          # the stop-and-go walk, with a pause on every marker
TEST_WALK    = "02_walk_normal_a"    # a continuous walk at a steady pace

---
## 1. From a recording to a map

The mapping walk is the only one where you stopped on the markers. Those pauses are what let us turn
**time** into **distance**: we know that during the 7th pause you were standing at 14 m.

The first plot finds the pauses. **It must find exactly 13** — one per marker. If it does not, change
`SENSITIVITY` and run again: lower means "only really still counts as a pause".

In [ ]:
SENSITIVITY = 0.5

plot_pauses(MAPPING_WALK, SENSITIVITY)

Now the map itself: the magnetic field along the path. This is your fingerprint of the corridor.

Note what the peaks and troughs line up with. And note that the first half of the corridor has only
wooden doors — yet it is not flat. You are not measuring the furniture, you are measuring the building.

In [ ]:
the_map = build_map(MAPPING_WALK, SENSITIVITY)

plot_map(the_map)

---
## 2. Is one reading enough?

Your phone reads a single value of |B|. Look at the map and ask where you could be.

Try a few values. Is there any reading in this corridor that identifies one single place?

In [ ]:
READING = 38.5     # microtesla

how_many_positions(the_map, READING)

---
## 3. Locating yourself

Here is the whole method, and it has no physics in it at all:

1. take **W metres** of your test walk,
2. slide that piece along the map,
3. wherever it fits best, that is the answer.

The left plot compares the answer to where you really were — points on the diagonal are correct.

**Start with `W = 0.1`.** That is a single reading, the case from section 2. Then try `W = 3`.

In [ ]:
W = 3.0            # length of the window, in METRES of corridor (not seconds!)
COMPARE = "values" # "values" compares the readings; "shape" ignores a constant offset

plot_location(the_map, load_walk(TEST_WALK), W, COMPARE)

So how long should the window be? Longer is always more accurate — but ask yourself what you are
actually being told when W is large.

In [ ]:
plot_error_vs_window(the_map, load_walk(TEST_WALK))

---
## 4. Where it fails, and why

The purple curve below is computed **from the map alone, with no test walk at all**. For every place, it
asks: *how similar is my best look-alike somewhere else in this corridor?* Low means "this place has a twin".

Compare it with where the method actually got it wrong.

In [ ]:
plot_ambiguity(the_map, load_walk(TEST_WALK))

---
## 5. The awkward walks

Same method, different walk. Predict what will happen before you run each one.

- `04_walk_fast` — you walked much faster.
- `05_walk_with_stop` — you stood still for a few seconds in the middle.
- `06_walk_pocket` — the phone was in your pocket.
- `09_walk_rotated` — the phone was held upside down.
- `07_walk_reverse` — you walked the path backwards.

In [ ]:
WALK = "05_walk_with_stop"

plot_location(the_map, load_walk(WALK), W=3.0)

---
## 6. What if you only walked part of it?

Everything so far assumed you walked the whole path, start to finish. But that was only needed to know
where you *really* were, so that we could measure the error.

The method itself needs no such thing. Below we take **one fragment** of a recording — a few seconds,
from anywhere — and pretend we know nothing: not where the walk started, not where it ended, not even how
fast you were going. We just guess a walking speed.

Try changing the guess. Then try changing the instant.

In [ ]:
AT_SECOND      = 20.0   # any moment inside the recording
ASSUMED_SPEED  = 0.7    # m/s — a guess. Try 0.5, try 1.2.

where_am_i(the_map, TEST_WALK, AT_SECOND, ASSUMED_SPEED)

---
## 7. Optional — what your phone is not telling you

Sensor Logger records two magnetometers: the calibrated one, and the raw one.

Change `MAGNETOMETER` below and look at the vertical axis of the map first. Then look at the error on the
rotated walk.

In [ ]:
MAGNETOMETER = "uncalibrated"    # or "calibrated"

raw_map = build_map(MAPPING_WALK, SENSITIVITY, MAGNETOMETER)
plot_map(raw_map)

for walk in ["02_walk_normal_a", "09_walk_rotated"]:
    e = error(raw_map, load_walk(walk, MAGNETOMETER), 3.0)
    print(f"{walk:20s}  median error {e:.2f} m")

---
## What to hand in (optional, for bonus credit)

This notebook run on **your own** recordings, plus short answers to four questions:

1. Looking at your own map, why is a single reading not enough?
2. What is the smallest window that gets you under 1 m of error? What do you lose by making it larger?
3. Where along the path does the method fail? Are the failures in the same places for two walks you did
   identically? What do those places have in common?
4. What happens in the walk where you stopped? Which is wrong there — the algorithm, or the thing we are
   comparing it against? What would you add to fix it?
